In [1]:
## 1. Imports and Paths

import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = "/content/drive/MyDrive/Master_datascience_thesis"

preprocessing_dir = os.path.join(PROJECT_ROOT, "outputs", "preprocessing")

traditional_ml_dir = os.path.join(PROJECT_ROOT, "outputs", "traditional_ml")

deep_learning_dir = os.path.join(PROJECT_ROOT, "outputs", "deep_learning")

model_selection_dir = os.path.join(PROJECT_ROOT, "outputs", "model_selection")

os.makedirs(model_selection_dir, exist_ok=True)

print("Notebook 06 setup complete")

Mounted at /content/drive
Notebook 06 setup complete


In [2]:
## 2. Load Validation Metadata and Model Results

validation_df = pd.read_csv(os.path.join(preprocessing_dir, "final_validation_metadata.csv"))

traditional_results = pd.read_csv(os.path.join(traditional_ml_dir,"traditional_ml_validation_results.csv"))

finetuned_dl_results = pd.read_csv(os.path.join(deep_learning_dir,"deep_learning_finetuned_validation_results.csv"))

all_default_results = pd.concat([traditional_results, finetuned_dl_results],ignore_index=True)

print("Validation images:", len(validation_df))

display(all_default_results)

Validation images: 1794


,Model,Accuracy,Precision,Sensitivity,Specificity,F1_score,AUROC,TN,FP,FN,TP
0,SVM,0.965440,0.979167,0.705,0.998118,0.819767,0.988363,1591,3,59,141
1,Random Forest,0.915273,1.000000,0.240,1.000000,0.387097,0.971948,1594,0,152,48
2,ResNet50,0.947046,0.689531,0.955,0.946048,0.800839,0.987312,1508,86,9,191
3,MobileNetV2,0.947603,0.719008,0.870,0.957340,0.787330,0.966662,1526,68,26,174


In [3]:
## 3. Load Continuous Validation Scores

traditional_scores = pd.read_csv(os.path.join(traditional_ml_dir,"traditional_ml_validation_scores.csv"))

finetuned_dl_scores = pd.read_csv(os.path.join(deep_learning_dir,"deep_learning_finetuned_validation_scores.csv"))

print("Traditional ML scores:", traditional_scores.shape)
print("Fine-tuned DL scores:", finetuned_dl_scores.shape)

display(traditional_scores.head())
display(finetuned_dl_scores.head())

Traditional ML scores: (1794, 4)
Fine-tuned DL scores: (1794, 4)


,path,label,SVM_score,Random_Forest_score
0,sick/s0008.png,0,-1.498901,0.02000
1,sick/s1596.png,0,-0.869189,0.33500
2,sick/s2013.png,0,-1.335010,0.01500
3,health/h1668.png,0,-1.033362,0.03625
4,sick/s3863.png,0,-0.687525,0.21625


,path,label,ResNet50_score,MobileNetV2_score
0,sick/s0008.png,0,0.000042,0.000216
1,sick/s1596.png,0,0.137693,0.060533
2,sick/s2013.png,0,0.001649,0.002398
3,health/h1668.png,0,0.005818,0.057680
4,sick/s3863.png,0,0.004497,0.022454


In [4]:
## 4. Check Path and Label Alignment

print("Traditional paths match validation:",np.array_equal(traditional_scores["path"].to_numpy(),validation_df["path"].to_numpy()))

print("DL paths match validation:",np.array_equal(finetuned_dl_scores["path"].to_numpy(),validation_df["path"].to_numpy()))

print("Traditional labels match validation:",np.array_equal(traditional_scores["label"].to_numpy(),validation_df["label"].to_numpy()))

print("DL labels match validation:",np.array_equal(finetuned_dl_scores["label"].to_numpy(),validation_df["label"].to_numpy()))

Traditional paths match validation: True
DL paths match validation: True
Traditional labels match validation: True
DL labels match validation: True


In [5]:
## 5. Find Thresholds for >=90% Sensitivity

from sklearn.metrics import roc_curve

def find_threshold(y_true, y_score):
    fpr, sensitivity, thresholds = roc_curve(y_true, y_score)

    valid = sensitivity >= 0.90

    best_index = np.argmin(fpr[valid])

    return thresholds[valid][best_index]


y_true = validation_df["label"].to_numpy()

svm_threshold = find_threshold(
    y_true,
    traditional_scores["SVM_score"].to_numpy()
)

rf_threshold = find_threshold(
    y_true,
    traditional_scores["Random_Forest_score"].to_numpy()
)

resnet50_threshold = find_threshold(
    y_true,
    finetuned_dl_scores["ResNet50_score"].to_numpy()
)

mobilenetv2_threshold = find_threshold(
    y_true,
    finetuned_dl_scores["MobileNetV2_score"].to_numpy()
)

print("SVM threshold:", svm_threshold)
print("Random Forest threshold:", rf_threshold)
print("ResNet50 threshold:", resnet50_threshold)
print("MobileNetV2 threshold:", mobilenetv2_threshold)

SVM threshold: -0.4910667285430168
Random Forest threshold: 0.14
ResNet50 threshold: 0.69848794
MobileNetV2 threshold: 0.37429488


In [6]:
## 6. Calculate Metrics at >=90% Sensitivity Thresholds

def metrics_at_threshold(y_true, scores, threshold):
    predictions = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, predictions).ravel()

    return {
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions),
        "Sensitivity": recall_score(y_true, predictions),
        "Specificity": tn / (tn + fp),
        "F1_score": f1_score(y_true, predictions),
        "AUROC": roc_auc_score(y_true, scores),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

threshold_results = pd.DataFrame([
    {
        "Model": "SVM",
        **metrics_at_threshold(
            y_true,
            traditional_scores["SVM_score"].to_numpy(),
            svm_threshold
        )
    },
    {
        "Model": "Random Forest",
        **metrics_at_threshold(
            y_true,
            traditional_scores["Random_Forest_score"].to_numpy(),
            rf_threshold
        )
    },
    {
        "Model": "ResNet50",
        **metrics_at_threshold(
            y_true,
            finetuned_dl_scores["ResNet50_score"].to_numpy(),
            resnet50_threshold
        )
    },
    {
        "Model": "MobileNetV2",
        **metrics_at_threshold(
            y_true,
            finetuned_dl_scores["MobileNetV2_score"].to_numpy(),
            mobilenetv2_threshold
        )
    }
])

display(threshold_results)

,Model,Threshold,Accuracy,Precision,Sensitivity,Specificity,F1_score,AUROC,TN,FP,FN,TP
0,SVM,-0.491067,0.962653,0.792952,0.9,0.970514,0.843091,0.988363,1547,47,20,180
1,Random Forest,0.140000,0.934225,0.647482,0.9,0.938519,0.753138,0.971948,1496,98,20,180
2,ResNet50,0.698488,0.959866,0.775862,0.9,0.967378,0.833333,0.987312,1542,52,20,180
3,MobileNetV2,0.374295,0.933668,0.645161,0.9,0.937892,0.751566,0.966662,1495,99,20,180


In [7]:
## 7. Save Final Model Selection

threshold_results.to_csv(os.path.join(model_selection_dir,"threshold_comparison.csv"),index=False)

selected_models = pd.DataFrame({
    "Model": ["SVM", "ResNet50"],
    "Role": [
        "Primary model",
        "Secondary model"
    ],
    "Threshold": [
        svm_threshold,
        resnet50_threshold
    ],
    "Model_file": [
        "best_svm_hog_model.pkl",
        "best_resnet50_finetuned.keras"
    ]
})

selected_models.to_csv(os.path.join(model_selection_dir,"selected_models.csv"),index=False)

display(selected_models)

,Model,Role,Threshold,Model_file
0,SVM,Primary model,-0.491067,best_svm_hog_model.pkl
1,ResNet50,Secondary model,0.698488,best_resnet50_finetuned.keras
